# 문제: 비행기 지연 예측

이 노트북의 목표는 다음과 같습니다.
- 다운로드한 .zip 파일에서 데이터 세트 처리 및 생성
- 탐색적 데이터 분석(EDA) 수행
- 기준 모델 마련
- 단순 모델에서 앙상블 모델로 진행
- 하이퍼파라미터 최적화 수행
- 특성 중요도 확인


## 비즈니스 시나리오 소개

여러분이 일하고 있는 여행 예약 웹 사이트에서는 항공편이 지연된 승객의 고객 경험을 개선하고자 합니다. 이 회사는 고객이 미국에서 국내 여행객들로 가장 혼잡한 공항에 이착륙하는 항공편을 예약할 때 날씨로 인해 항공편이 지연되는지 고객에게 알려줄 수 있는 기능을 만들려고 합니다. 

기계 학습(ML)을 이용하여 날씨로 인해 항공편이 지연될지 여부를 파악함으로써 이 문제의 일부를 해결해야 합니다. 여러분에게는 대형 항공사가 운영하는 국내선의 정시운항성 데이터 세트에 액세스할 수 있는 권한이 부여되었습니다. 이 데이터를 사용하여 가장 혼잡한 공항의 항공편이 지연될지 예측하는 기계 학습 모델을 훈련할 수 있습니다.


## 데이터 세트 소개

이 데이터 세트에는 국내 예약 승객 매출의 1% 이상을 차지하는 미국 공인 항공사에서 보고한 예정 및 실제 출발 및 도착 시간이 포함되어 있습니다. 데이터는 미국 교통 통계국(BTS) 항공사 정보실에서 수집했습니다. 이 데이터 세트에는 2013년부터 2018년 사이 항공편의 날짜, 시간, 출발지, 목적지, 항공사, 거리 및 지연 상태가 포함되어 있습니다.


### 특성

이 데이터 세트의 특성에 대한 자세한 내용은 [On-time delay dataset features](https://www.transtats.bts.gov/Fields.asp)를 참조하십시오.

### 데이터 세트 속성  
웹 사이트: https://www.transtats.bts.gov/

이 실습에서 사용된 데이터 세트는 미국 미국 교통 통계국(BTS) 항공사 정보실에서 수집한 항공편 정시운항성 데이터를 컴파일한 것으로 https://www.transtats.bts.gov/DatabaseInfo.asp?DB_ID=120&amp;DB_URL=Mode_ID=1&amp;Mode_Desc=Aviation&amp;Subject_ID2=0에서 확인할 수 있습니다.

# 1단계: 문제 공식화 및 데이터 수집

이 시나리오의 비즈니스 문제와 달성하고자 하는 비즈니스 목표를 몇 문장으로 요약하여 작성하는 것으로 이 프로젝트를 시작하십시오. 다음 섹션에 아이디어를 적어 둘 수 있습니다. 팀에서 지향했으면 하는 비즈니스 지표를 포함합니다. 해당 정보를 정의한 후 기계 학습 문제 기술서를 작성합니다. 마지막으로, 이러한 문제에 해당하는 기계 학습 활동의 유형에 대해 설명을 한 두 개 추가합니다. 

#### <span style="color: blue;">프로젝트 프레젠테이션: 프로젝트 프레젠테이션에 이러한 세부 정보에 대한 요약을 포함합니다.</span>

### 1. 기계 학습이 이 시나리오에 배포하기 적합한 솔루션인지와 그 이유 파악

In [ ]:
# Write your answer here

### 2. 비즈니스 문제, 성공 지표 및 원하는 기계 학습 결과 공식화

In [ ]:
# Write your answer here

### 3. 작업 중인 기계 학습 문제의 유형 식별

In [ ]:
# Write your answer here

### 4. 작업 중인 데이터의 적합성 분석

In [ ]:
# Write your answer here

### 설정

이제 어디에 집중할지 결정했으므로 문제 해결을 시작할 수 있도록 이 실습을 설정하겠습니다.

**참고:** 이 노트북은 스토리지 25GB를 보유한 `ml.m4.xlarge` 노트북 인스턴스에서 생성 및 테스트되었습니다. 

In [ ]:
import os
from pathlib2 import Path
from zipfile import ZipFile
import time

import pandas as pd
import numpy as np
import subprocess

import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
instance_type='ml.m4.xlarge'

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# 2단계: 데이터 전처리 및 시각화  
이 데이터 전처리 단계에서는 데이터를 더 잘 이해할 수 있도록 데이터를 탐색하고 시각화할 수 있습니다. 먼저 필요한 라이브러리를 가져와서 데이터를 pandas DataFrame으로 읽어 들입니다. 데이터를 가져온 후 데이터 세트를 탐색합니다. 데이터 세트의 형태를 찾고 열과 작업할 열 유형(숫자형, 범주형)을 탐색합니다. 특성에 대한 기본 통계를 수행하여 특성의 평균 및 범위를 파악하는 것이 좋습니다. 대상 열을 면밀히 검토하고 해당 분포를 확인합니다.


### 고려해야 할 구체적인 질문

이 실습 섹션을 진행하면서 다음 질문을 고려해보십시오.

1. 특성에 대해 실행한 기본 통계에서 추론할 수 있는 것은 무엇인가요? 
2. 대상 클래스의 분포에서 추론할 수 있는 것은 무엇인가요?
3. 데이터를 탐색하면서 추론할 수 있는 것이 또 있나요?

#### <span style="color: blue;">프로젝트 프레젠테이션: 이러한 질문(및 기타 유사한 질문)에 대한 답변을 요약하여 프로젝트 프레젠테이션에 포함합니다.</span>

먼저 퍼블릭 Amazon Simple Storage Service(Amazon S3) 버킷에서 이 노트북 환경으로 데이터 세트를 가져옵니다.

In [ ]:
# download the files

zip_path = '/home/ec2-user/SageMaker/project/data/FlightDelays/'
base_path = '/home/ec2-user/SageMaker/project/data/FlightDelays/'
csv_base_path = '/home/ec2-user/SageMaker/project/data/csvFlightDelays/'

!mkdir -p {zip_path}
!mkdir -p {csv_base_path}
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMLFO-1/flight_delay_project/data/ {zip_path} --recursive


In [ ]:
zip_files = [str(file) for file in list(Path(base_path).iterdir()) if '.zip' in str(file)]
len(zip_files)

.zip 파일에서 쉼표로 구분된 값(CSV) 파일을 추출합니다.

In [ ]:
def zip2csv(zipFile_name , file_path):
    """
    Extract csv from zip files
    zipFile_name: name of the zip file
    file_path : name of the folder to store csv
    """

    try:
        with ZipFile(zipFile_name, 'r') as z: 
            print(f'Extracting {zipFile_name} ') 
            z.extractall(path=file_path) 
    except:
        print(f'zip2csv failed for {zipFile_name}')

for file in zip_files:
    zip2csv(file, csv_base_path)

print("Files Extracted")

In [ ]:
csv_files = [str(file) for file in list(Path(csv_base_path).iterdir()) if '.csv' in str(file)]
len(csv_files)

CSV 파일을 로드하기 전에 추출된 폴더에서 HTML 파일을 읽습니다. 이 HTML 파일에는 데이터 세트에 포함된 특성에 대한 배경과 추가 정보가 포함되어 있습니다.

In [ ]:
from IPython.display import IFrame

IFrame(src=os.path.relpath(f"{csv_base_path}readme.html"), width=1000, height=600)

#### 샘플 CSV 파일 다운로드

모든 CSV 파일을 결합하기 전에 단일 CSV 파일에서 데이터를 검사하십시오. pandas를 사용하여 먼저 `On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_9.csv` 파일을 읽으십시오. Python에서 기본으로 제공하는 `read_csv` 함수를 사용할 수 있습니다([pandas.read_csv 설명서](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)).

In [ ]:
df_temp = pd.read_csv(f"{csv_base_path}On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_9.csv")

**질문**: 데이터 세트의 행 및 열 길이를 출력하고 열 이름을 출력합니다.

**힌트**: DataFrame의 행과 열을 보려면 `<DataFrame>.shape` 함수를 사용합니다. 열 이름을 보려면 `<DataFrame>.columns` 함수를 사용합니다.

In [ ]:
df_shape = # **ENTER YOUR CODE HERE**
print(f'Rows and columns in one CSV file is {df_shape}')

**질문**: 데이터 세트의 처음 10개 행을 출력합니다.  

**힌트**: `x`개의 행을 출력하려면 pandas에서 기본 제공하는 `head(x)` 함수를 사용하십시오.

In [ ]:
# Enter your code here

**질문**: 데이터 세트의 모든 열을 출력합니다. 열 이름을 보려면 `<DataFrame>.columns`를 사용합니다.

In [ ]:
print(f'The column names are :')
print('#########')
for col in <CODE>:# **ENTER YOUR CODE HERE**
    print(col)

**질문**: 데이터 세트에서 *Del*이라는 단어가 포함된 열을 모두 출력합니다. 이렇게 하면 *지연 데이터*가 있는 열의 수를 확인할 수 있습니다.

**힌트**: 특정 `if` 문 기준을 통과하는 값을 포함하려면 Python 리스트 컴프리헨션을 사용합니다.

예: `[x for x in [1,2,3,4,5] if x > 2]`  

**힌트**: 값이 리스트에 있는지 확인하려면 `in` 키워드([키워드 설명서로 본 Python](https://www.w3schools.com/python/ref_keyword_in.asp))를 사용하면 됩니다. 

예: `5 in [1,2,3,4,5]`

In [ ]:
# Enter your code here

다음은 데이터 세트에 대해 자세히 배워보는 데 도움이 되는 몇 가지 추가 질문입니다.

**질문**   

1. 데이터 세트에는 행과 열이 몇 개 있나요?   
2. 데이터 세트에는 몇 년이 포함되어 있나요?   
3. 데이터 세트의 날짜 범위는 어떻게 되나요?   
4. 데이터 세트에는 어떤 항공사가 포함되었나요?   
5. 포함된 출발지 공항과 목적지 공항은 어디인가요?

**힌트**
- `df_temp.shape` 사용을 통해 DataFrame의 크기를 표시합니다.
- `df_temp.columnName`(예: `df_temp.CarrierDelay`) 사용을 통해 특정 열을 참조합니다.
- `df_temp.column.unique()`(예: `df_temp.Year.unique()`) 사용을 통해 열에 대한 고유 값을 얻습니다.

In [ ]:
print("The #rows and #columns are ", <CODE> , " and ", <CODE>)
print("The years in this dataset are: ", <CODE>)
print("The months covered in this dataset are: ", <CODE>)
print("The date range for data is :" , min(<CODE>), " to ", max(<CODE>))
print("The airlines covered in this dataset are: ", list(<CODE>))
print("The Origin airports covered are: ", list(<CODE>))
print("The Destination airports covered are: ", list(<CODE>))

**질문**: 모든 출발지 및 도착지 공항의 수는 어떻게 되나요?

**힌트**: **Origin** 및 **Dest** 열을 사용하여 각 공항의 값을 찾으려면 pandas에서 `values_count` 함수를 사용하면 됩니다([pandas.Series.value_counts 설명서](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)).

In [ ]:
counts = pd.DataFrame({'Origin':<CODE>, 'Destination':<CODE>})
counts

**질문**: 데이터 세트의 항공편 수를 기준으로 상위 15개의 출발지 및 목적지 공항을 출력합니다.

**힌트**: Pandas에서 `sort_values` 함수를 사용하면 됩니다([pandas.DataFrame.sort_values 설명서](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html)).

In [ ]:
counts.sort_values(by=<CODE>,ascending=False).head(15) # Enter your code here

**비행 여정에 대한 모든 정보를 종합해 볼 때 연착 여부를 예측할 수 있을까요?**

**ArrDel15** 열은 15분 이상 지연될 때 *1*의 값을 취하는 표시 변수입니다. 그렇지 않으면, *0*의 값을 취합니다.

이 열을 분류 문제의 대상 열로 사용할 수 있습니다.

이제 샌프란시스코에서 로스앤젤레스로 출장을 간다고 가정해 보겠습니다. 로스앤젤레스 예약을 더 잘 관리하고자 합니다. 따라서, 주어진 특성 세트를 활용하여 항공편이 지연될지에 관한 아이디어를 얻고 싶습니다. 이 데이터 세트에서 비행 전에 알아야 할 특성은 몇 가지인가요?

`DepDelay`, `ArrDelay`, `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`, `DivArrDelay`열에는 지연에 관한 정보가 포함되어 있습니다. 하지만 이러한 지연은 출발지 또는 목적지 어디서나 발생할 수 있습니다. 착륙 10분 전에 갑작스러운 기상 악화로 지연되는 경우 이 데이터는 로스앤젤레스의 예약을 관리하는 데 도움이 되지 않습니다.

따라서 문제 기술을 간소화하기 위해 다음 열을 고려하여 도착 지연을 예측하십시오.<br>

`Year`, `Quarter`, `Month`, `DayofMonth`, `DayOfWeek`, `FlightDate`, `Reporting_Airline`, `Origin`, `OriginState`, `Dest`, `DestState`, `CRSDepTime`, `DepDelayMinutes`, `DepartureDelayGroups`, `Cancelled`, `Diverted`, `Distance`, `DistanceGroup`, `ArrDelay`, `ArrDelayMinutes`, `ArrDel15`, `AirTime`

또한 출발지 및 목적지 공항을 다음과 같이 필터링합니다.
- 주요 공항: ATL, ORD, DFW, DEN, CLT, LAX, IAH, PHX, SFO
- 상위 5개 항공사: UA, OO, WN, AA, DL

이 정보는 결합될 CSV 파일의 데이터 크기를 줄이는 데 도움이 됩니다.

#### 모든 CSV 파일 결합
 
먼저 각 파일에서 개별 DataFrame을 복사하는 데 사용할 빈 DataFrame을 생성합니다. 그런 다음 `csv_files` 리스트의 각 파일에 대해 다음을 수행합니다.

1. CSV 파일을 DataFrame으로 읽어 들입니다. 
2. `filter_cols` 변수를 기반으로 열을 필터링합니다.

```
        columns = ['col1', 'col2']
        df_filter = df[columns]
```

3. 각 `subset_cols`에서 `subset_vals`만 유지합니다. `val`이 DataFrame 열에 있는지 확인하려면 pandas ([pandas.DataFram.isin 설명서] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isin.html))에서 `isin` 함수를 사용합니다. 그런 다음 이를 포함하는 행을 선택합니다.

```
        df_eg[df_eg['col1'].isin('5')]
```

4. DataFrame과 빈 DataFrame을 연결합니다. 

In [ ]:
def combine_csv(csv_files, filter_cols, subset_cols, subset_vals, file_name):

    """
    Combine csv files into one Data Frame
    csv_files: list of csv file paths
    filter_cols: list of columns to filter
    subset_cols: list of columns to subset rows
    subset_vals: list of list of values to subset rows
    """

    df = pd.DataFrame()
    
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_temp = df_temp[filter_cols]
        for col, val in zip(subset_cols,subset_vals):
            df_temp = df_temp[df_temp[col].isin(val)]      
        
        df = pd.concat([df, df_temp], axis=0)
      
    df.to_csv(file_name, index=False)
    print(f'Combined csv stored at {file_name}')

In [ ]:
#cols is the list of columns to predict Arrival Delay 
cols = ['Year','Quarter','Month','DayofMonth','DayOfWeek','FlightDate',
        'Reporting_Airline','Origin','OriginState','Dest','DestState',
        'CRSDepTime','Cancelled','Diverted','Distance','DistanceGroup',
        'ArrDelay','ArrDelayMinutes','ArrDel15','AirTime']

subset_cols = ['Origin', 'Dest', 'Reporting_Airline']

# subset_vals is a list collection of the top origin and destination airports and top 5 airlines
subset_vals = [['ATL', 'ORD', 'DFW', 'DEN', 'CLT', 'LAX', 'IAH', 'PHX', 'SFO'], 
               ['ATL', 'ORD', 'DFW', 'DEN', 'CLT', 'LAX', 'IAH', 'PHX', 'SFO'], 
               ['UA', 'OO', 'WN', 'AA', 'DL']]

이전의 함수를 사용하여 다른 모든 파일을 쉽게 읽을 수 있는 단일 파일로 병합할 수 있습니다. 

**참고**: 이 프로세스를 완료하는 데 5~7분이 걸립니다.

In [ ]:
start = time.time()
combined_csv_filename = f"{base_path}combined_files.csv"
combine_csv(csv_files, cols, subset_cols, subset_vals, combined_csv_filename)
print(f'CSVs merged in {round((time.time() - start)/60,2)} minutes')

#### 데이터 세트 로드

결합된 데이터 세트를 로드합니다.

In [ ]:
data = pd.read_csv(combined_csv_filename)

처음 다섯 개의 레코드를 출력합니다.

In [ ]:
# Enter your code here 

다음은 데이터 세트에 대해 자세히 배워보는 데 도움이 되는 몇 가지 추가 질문입니다.

**질문**   

1. 데이터 세트에는 행과 열이 몇 개 있나요?   
2. 데이터 세트에는 몇 년이 포함되어 있나요?   
3. 데이터 세트의 날짜 범위는 어떻게 되나요?   
4. 데이터 세트에는 어떤 항공사가 포함되었나요?   
5. 포함된 출발지 공항과 목적지 공항은 어디인가요?

In [ ]:
print("The #rows and #columns are ", <CODE> , " and ", <CODE>)
print("The years in this dataset are: ", list(<CODE>))
print("The months covered in this dataset are: ", sorted(list(<CODE>)))
print("The date range for data is :" , min(<CODE>), " to ", max(<CODE>))
print("The airlines covered in this dataset are: ", list(<CODE>))
print("The Origin airports covered are: ", list(<CODE>))
print("The Destination airports covered are: ", list(<CODE>))

**is_delay**(*1*은 도착 시간이 15분 이상 지연되었음을 의미하고 *0*은 그 외의 모든 경우를 의미합니다)로 대상 열을 정의합니다. `rename` 메서드를 사용하여 열 이름을 **ArrDel15**에서 *is_delay*로 바꿉니다.

**힌트**: pandas에서 `rename` 함수를 사용하면 됩니다([pandas.DataFrame.rename 설명서](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html)).

예:
```
data.rename(columns={'col1':'column1'}, inplace=True)
```

In [ ]:
data.rename(columns=<CODE>, inplace=True) # Enter your code here

열에서 null을 찾습니다. `isnull()` 함수를 사용하면 됩니다([pandas.isnull 설명서](https://pandas.pydata.org/pandas-docs/version/0.17.0/generated/pandas.isnull.html)).

**힌트**: `isnull()`의 역할은 특정 값이 null인지 여부를 감지하는 것입니다. 그것은 그 자리에 부울(*True* 또는 *False*)을 반환합니다. 열 수를 합산하려면 `sum(axis=0)` 함수를 사용합니다(예: `df.isnull().sum(axis=0)`).

In [ ]:
# Enter your code here

1,658,130행 중 22,540개(1.3%)에 대한 도착 지연 세부 정보 및 비행 시간이 누락되었습니다. 이러한 행을 제거하거나 대치할 수 있습니다. 설명서에는 누락된 행에 대한 정보를 언급하지 않고 있습니다.


In [ ]:
### Remove null columns
data = data[~data.is_delay.isnull()]
data.isnull().sum(axis = 0)

CRSDepTime에서 24시간 형식으로 시간을 가져옵니다.

In [ ]:
data['DepHourofDay'] = (data['CRSDepTime']//100)

## **기계 학습 문제 기술서**
- 일련의 특성을 감안할 때 항공편이 15분 넘게 지연될 것인지 예측할 수 있나요?
- 대상 변수는 *0* 또는 *1* 값만 사용하므로 분류 알고리즘을 사용할 수 있습니다. 

모델링을 시작하기 전에 특성 분포, 상관 관계 등을 살펴보는 것이 좋습니다.
- 데이터의 비선형성 또는 패턴을 파악할 수 있습니다.
    - 선형 모델: 거듭제곱, 지수 또는 상호 작용 특성 추가
    - 비선형 모델 시도
- 데이터 불균형 
    - 편향된 모델 성능을 제공하지 않는 지표 선택(정확도와 곡선 아래 면적, 즉 AUC 비교)
    - 가중치 또는 사용자 지정 손실 함수 사용
- 누락된 데이터
    - 평균, 중앙값, 모드(숫자형 변수), 빈도 클래스(범주형 변수)와 같은 단순 통계를 기반으로 대체 수행
    - 군집화 기반 대체(열 값을 예측하는 K-최근접 이웃 알고리즘, 즉 KNN)
    - 열 삭제

### 데이터 탐색

*지연*과 *지연 없음* 클래스를 비교 확인하십시오.


In [ ]:
(data.groupby('is_delay').size()/len(data) ).plot(kind='bar')# Enter your code here
plt.ylabel('Frequency')
plt.title('Distribution of classes')
plt.show()

**질문**: *지연*과 *지연 없음*의 비율에 대한 막대 플롯 비교를 통해 무엇을 추론할 수 있나요?

In [ ]:
# Enter your answer here

다음 셀을 실행하고 질문에 답변하십시오.

In [ ]:
viz_columns = ['Month', 'DepHourofDay', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest']
fig, axes = plt.subplots(3, 2, figsize=(20,20), squeeze=False)
# fig.autofmt_xdate(rotation=90)

for idx, column in enumerate(viz_columns):
    ax = axes[idx//2, idx%2]
    temp = data.groupby(column)['is_delay'].value_counts(normalize=True).rename('percentage').\
    mul(100).reset_index().sort_values(column)
    sns.barplot(x=column, y="percentage", hue="is_delay", data=temp, ax=ax)
    plt.ylabel('% delay/no-delay')
    

plt.show()

In [ ]:
sns.lmplot( x="is_delay", y="Distance", data=data, fit_reg=False, hue='is_delay', legend=False)
plt.legend(loc='center')
plt.xlabel('is_delay')
plt.ylabel('Distance')
plt.show()

**질문**

이전 차트의 데이터를 사용하여 다음 질문에 답하십시오.

- 지연이 가장 많이 발생하는 달은 언제인가요?
- 하루 중 언제 가장 지연이 많이 발생하나요?
- 일주일 중 언제 가장 지연이 많이 발생하나요?
- 가장 많이 지연되는 항공사는 어디인가요?
- 가장 많이 지연되는 출발지 및 목적지 공항은 어디인가요?
- 비행 거리가 지연에 영향을 미치는 요인인가요?

In [ ]:
# Enter your answers here

### 특성

모든 열과 각 열의 특정 유형을 살펴봅니다.

In [ ]:
data.columns

In [ ]:
data.dtypes

필요한 열 필터링하기
- 날짜를 설명하는 *Year*, *Quarter*, *Month*, *DayofMonth* 및 *DayOfWeek* 열이 있으므로 *Date*가 중복됩니다.
- *OriginState* 및 *DestState* 대신 *Origin* 및 *Dest* 코드를 사용합니다.
- 항공편 지연 여부만 분류하는 것이므로 *TotalDelayMinutes*, *DepDelayMinutes*, *ArrDelayMinutes*는 필요하지 않습니다.

*DepHourofDay*는 대상과 정량적 관계가 없으므로 범주형 변수로 취급합니다.
- 이 변수를 원-핫 인코딩해야 하는 경우 23개의 열이 추가로 생성됩니다.
- 범주형 변수를 처리하는 다른 대안으로는 해시 인코딩, 정규화된 평균 인코딩, 값 버킷화 등이 있습니다.
- 이 경우 버킷으로 나누기만 하면 됩니다.

열 유형을 범주로 변경하려면 `astype` 함수([pandas.DataFrame.astype 설명서](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.astype.html))를 사용합니다.

In [ ]:
data_orig = data.copy()
data = data[[ 'is_delay', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest','Distance','DepHourofDay']]
categorical_columns  = ['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest', 'DepHourofDay']
for c in categorical_columns:
    data[c] = data[c].astype('category')

원-핫 인코딩을 사용하려면 선택한 범주 열에 pandas `get_dummies` 함수를 사용합니다. 그런 다음 생성된 특성을 pandas `concat` 함수를 사용하여 원래 데이터 세트에 연결할 수 있습니다. 범주형 변수를 인코딩하는 경우 `drop_first=True` 키워드로 *더미 인코딩*을 수행할 수도 있습니다. 더미 인코딩에 대한 자세한 내용은 [더미 변수 (통계)] (https://en.wikiversity.org/wiki/Dummy_variable_(statistics))를 참조하십시오.

예:
```
pd.get_dummies(df[['column1','columns2']], drop_first=True)
```

In [ ]:
data_dummies = pd.get_dummies(<CODE>, drop_first=True) # Enter your code here
data = pd.concat([<CODE>, <CODE>], axis = 1)
data.drop(categorical_columns,axis=1, inplace=True)

데이터 세트의 길이와 새로운 열을 확인합니다.

**힌트**: `shape` 및 `columns` 속성을 사용합니다.

In [ ]:
# Enter your code here

In [ ]:
# Enter your code here

이제 모델을 훈련할 준비가 되었습니다. 데이터를 분할하기 전에 **is_delay** 열의 이름을 *target*으로 바꿉니다.

**힌트**: pandas에서 `rename` 함수를 사용하면 됩니다([pandas.DataFrame.rename 설명서](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html)).

In [ ]:
data.rename(columns = {<CODE>:<CODE>}, inplace=True )# Enter your code here

## <span style="color:red"> 2단계 종료 </span>

프로젝트 파일을 로컬 컴퓨터에 저장합니다. 방법은 다음과 같습니다.

1. 왼쪽의 파일 탐색기에서 작업 중인 노트북을 마우스 오른쪽 단추로 클릭합니다. 

2. **다운로드**를 선택하고 파일을 로컬에 저장합니다.  

이 작업을 수행하면 현재 노트북이 컴퓨터의 기본 다운로드 폴더로 다운로드됩니다.

# 3단계: 모델 훈련 및 평가

DataFrame에서 기계 학습 알고리즘이 사용할 수 있는 형식으로 데이터 세트를 변환할 때 포함해야 하는 몇 가지 예비 단계가 있습니다. Amazon SageMaker에서는 다음 단계를 수행해야 합니다.

1. `sklearn.model_selection.train_test_split` 를 사용해 데이터를 `train_data`, `validation_data`, `test_data`로 분할합니다.  

2. 데이터 세트를 Amazon SageMaker 훈련 작업에서 사용할 수 있는 적절한 파일 형식으로 변환합니다. CSV 파일 또는 레코드 protobuf로 변환하면 됩니다. 자세한 내용은 [훈련을 위한 공통 데이터 형식](https://docs.aws.amazon.com/sagemaker/latest/dg/cdf-training.html) 페이지를 참조하십시오.  

3. 데이터를 S3 버킷에 업로드합니다. 이전에 버킷을 생성한 적이 없는 경우 [버킷 생성](https://docs.aws.amazon.com/AmazonS3/latest/gsg/CreatingABucket.html) 페이지를 참조하십시오.  

다음 셀을 사용하여 이러한 단계를 완료합니다. 필요에 따라 셀을 삽입하고 삭제합니다.

#### <span style="color: blue;">프로젝트 프레젠테이션: 프로젝트 프레젠테이션 시, 이 단계에서 내린 주요 결정을 적어 둡니다.</span>

### 훈련-테스트 분할

In [ ]:
from sklearn.model_selection import train_test_split
def split_data(data):
    train, test_and_validate = train_test_split(data, test_size=0.2, random_state=42, stratify=data['target'])
    test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['target'])
    return train, validate, test

In [ ]:
train, validate, test = split_data(data)
print(train['target'].value_counts())
print(test['target'].value_counts())
print(validate['target'].value_counts())

**샘플 답변**
```
0.0 1033570
1.0 274902
이름: 대상, dtype: int64
0.0 129076
1.0 34483
이름: 대상, dtype: int64
0.0 129612
1.0 33947
이름: 대상, dtype: int64
```

### 기준 분류 모델

In [ ]:
import sagemaker
from sagemaker.serializers import CSVSerializer
from sagemaker.amazon.amazon_estimator import RecordSet
import boto3

# Instantiate the LinearLearner estimator object with 1 ml.m4.xlarge
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                               instance_count=<CODE>,
                                               instance_type=<CODE>,
                                               predictor_type=<CODE>,
                                               binary_classifier_model_selection_criteria=<CODE>)

### 샘플 코드
```
num_classes = len(pd.unique(train_labels))
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                              instance_count=1,
                                              instance_type='ml.m4.xlarge',
                                              predictor_type='binary_classifier',
                                              binary_classifier_model_selection_criteria = 'cross_entropy_loss')
                                              
```

선형 학습자는 protobuf 또는 CSV 콘텐츠 유형의 훈련 데이터를 적용합니다. 또한 protobuf, CSV 또는 JavaScript Object Notation(JSON) 콘텐츠 유형의 추론 요청을 허용합니다. 훈련 데이터에는 특성과 ground-truth 레이블이 있으나, 추론 요청의 데이터에는 특성만 있습니다.

AWS는 프로덕션 파이프라인에서는 데이터를 Amazon SageMaker protobuf 형식으로 변환하여 Amazon S3에 저장할 것을 권장합니다. 신속한 시작과 실행을 위해, AWS는 데이터 세트가 로컬 메모리에 들어갈 수 있을 만큼 작은 경우 데이터 세트를 변환하고 업로드하는 `record_set` 작업을 제공합니다. 이 메서드는 이 단계에서 사용하는 것과 같은 NumPy 배열을 수락하므로 여기에서 사용해 보겠습니다. `RecordSet` 객체는 데이터의 임시 Amazon S3 위치를 추적합니다. `estimator.record_set` 함수를 사용하여 훈련, 검증 및 테스트 레코드를 만듭니다. 그런 다음 `estimator.fit` 기능을 사용하여 훈련 작업을 시작합니다.

In [ ]:
### Create train, validate, and test records
train_records = classifier_estimator.record_set(train.values[:, 1:].astype(np.float32), train.values[:, 0].astype(np.float32), channel='train')
val_records = classifier_estimator.record_set(validate.values[:, 1:].astype(np.float32), validate.values[:, 0].astype(np.float32), channel='validation')
test_records = classifier_estimator.record_set(test.values[:, 1:].astype(np.float32), test.values[:, 0].astype(np.float32), channel='test')

이제 방금 업로드한 데이터 세트로 모델을 훈련합니다.

### 샘플 코드
```
linear.fit([train_records,val_records,test_records])
```

In [ ]:
### Fit the classifier
# Enter your code here

## 모델 평가
이 섹션에서는 훈련된 모델을 평가합니다. 

먼저 훈련 작업에 대한 지표를 검토합니다.

In [ ]:
sagemaker.analytics.TrainingJobAnalytics(classifier_estimator._current_job_name, 
                                         metric_names = ['test:objective_loss', 
                                                         'test:binary_f_beta',
                                                         'test:precision',
                                                         'test:recall']
                                        ).dataframe()

그런 다음 테스트 데이터를 Amazon S3로 로드하고 배치 예측 함수를 사용하여 예측을 수행하는 데 도움이 되는 몇 가지 함수를 설정합니다. 배치 예측을 사용하면 제공된 테스트 데이터에 대해 예측이 수행되는 경우에만 인스턴스가 실행되므로 비용을 절감할 수 있습니다.

**참고: ** 실습 설정 중에 만든 실습용 버킷의 이름을 `<LabBucketName>` (으)로 바꿉니다.

In [ ]:
import io
#bucket='<LabBucketName>'
prefix='flight-linear'
train_file='flight_train.csv'
test_file='flight_test.csv'
validate_file='flight_validate.csv'
whole_file='flight.csv'
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False )
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

In [ ]:
def batch_linear_predict(test_data, estimator):
    batch_X = test_data.iloc[:,1:];
    batch_X_file='batch-in.csv'
    upload_s3_csv(batch_X_file, 'batch-in', batch_X)

    batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
    batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

    classifier_transformer = estimator.transformer(instance_count=1,
                                           instance_type='ml.m4.xlarge',
                                           strategy='MultiRecord',
                                           assemble_with='Line',
                                           output_path=batch_output)

    classifier_transformer.transform(data=batch_input,
                             data_type='S3Prefix',
                             content_type='text/csv',
                             split_type='Line')
    
    classifier_transformer.wait()

    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
    target_predicted_df = pd.read_json(io.BytesIO(obj['Body'].read()),orient="records",lines=True)
    return test_data.iloc[:,0], target_predicted_df.iloc[:,0]


테스트 데이터 세트에서 예측을 실행하려면 테스트 데이터 세트에서 (이전에 정의한) `batch_linear_predict` 함수를 실행합니다.


In [ ]:
test_labels, target_predicted = batch_linear_predict(test, classifier_estimator)

혼동 행렬 및 다양한 점수 지표에 관한 플롯을 보려면 다음과 같은 몇 가지 함수를 생성합니다.

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(test_labels, target_predicted):
    matrix = confusion_matrix(test_labels, target_predicted)
    df_confusion = pd.DataFrame(matrix)
    colormap = sns.color_palette("BrBG", 10)
    sns.heatmap(df_confusion, annot=True, fmt='.2f', cbar=None, cmap=colormap)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.ylabel("True Class")
    plt.xlabel("Predicted Class")
    plt.show()
    

In [ ]:
from sklearn import metrics

def plot_roc(test_labels, target_predicted):
    TN, FP, FN, TP = confusion_matrix(test_labels, target_predicted).ravel()
    # Sensitivity, hit rate, recall, or true positive rate
    Sensitivity  = float(TP)/(TP+FN)*100
    # Specificity or true negative rate
    Specificity  = float(TN)/(TN+FP)*100
    # Precision or positive predictive value
    Precision = float(TP)/(TP+FP)*100
    # Negative predictive value
    NPV = float(TN)/(TN+FN)*100
    # Fall out or false positive rate
    FPR = float(FP)/(FP+TN)*100
    # False negative rate
    FNR = float(FN)/(TP+FN)*100
    # False discovery rate
    FDR = float(FP)/(TP+FP)*100
    # Overall accuracy
    ACC = float(TP+TN)/(TP+FP+FN+TN)*100

    print("Sensitivity or TPR: ", Sensitivity, "%") 
    print( "Specificity or TNR: ",Specificity, "%") 
    print("Precision: ",Precision, "%") 
    print("Negative Predictive Value: ",NPV, "%") 
    print( "False Positive Rate: ",FPR,"%")
    print("False Negative Rate: ",FNR, "%") 
    print("False Discovery Rate: ",FDR, "%" )
    print("Accuracy: ",ACC, "%") 

    test_labels = test.iloc[:,0];
    print("Validation AUC", metrics.roc_auc_score(test_labels, target_predicted) )

    fpr, tpr, thresholds = metrics.roc_curve(test_labels, target_predicted)
    roc_auc = metrics.auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % (roc_auc))
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver operating characteristic')
    plt.legend(loc="lower right")

    # create the axis of thresholds (scores)
    ax2 = plt.gca().twinx()
    ax2.plot(fpr, thresholds, markeredgecolor='r',linestyle='dashed', color='r')
    ax2.set_ylabel('Threshold',color='r')
    ax2.set_ylim([thresholds[-1],thresholds[0]])
    ax2.set_xlim([fpr[0],fpr[-1]])

    print(plt.figure())

혼동 행렬을 플롯하려면 배치 작업의 `test_labels` 및 `target_predicted` 데이터에 대한 `plot_confusion_matrix` 함수를 호출합니다.

In [ ]:
# Enter your code here

통계를 출력하고 수신자 조작 특성(ROC) 곡선을 플롯하려면 배치 작업의 `test_labels` 및 `target_predicted` 데이터에 대한 `plot_roc` 함수를 호출합니다.

In [ ]:
# Enter your code here

### 고려해야 할 주요 질문

1. 훈련 세트의 성능과 비교하여 테스트 세트에서의 모델 성능이 어떤가요? 이 비교에서 무엇을 추론할 수 있나요? 
2. 정확도, 정밀도, 재현율과 같은 지표의 결과 간에 분명한 차이가 있나요? 만약 그렇다면, 이러한 차이가 나타나는 이유는 무엇일까요? 
3. 비즈니스 상황과 목표를 고려해 볼 때 고려해야 할 가장 중요한 지표는 무엇일까요? 이유가 무엇인가요?
4. 비즈니스 관점에서 가장 중요하다고 생각하는 지표의 결과가 필요한 결과로도 충분한가요? 그렇지 않다면 다음 반복 시 변경할 수 있는 사항은 무엇입니까? (이 작업은 다음 특성 엔지니어링 섹션에서 수행하게 됩니다.)

다음 셀을 사용하여 이러한 질문과 기타 질문에 답하십시오. 필요에 따라 셀을 삽입하고 삭제합니다.

#### <span style="color: blue;">프로젝트 프레젠테이션: 프로젝트 프레젠테이션에서 이러한 질문에 대한 답변과 답변할 수 있는 기타 유사한 질문들을 이 섹션에 적어 둡니다. 사용자가 내린 주요 세부 정보 및 결정을 기록합니다.</span>


**질문**: 혼동 행렬에서 요약할 수 있는 것은 무엇인가요?


In [ ]:
# Enter your answer here

## <span style="color:red"> 3단계 종료 </span>

프로젝트 파일을 로컬 컴퓨터에 저장합니다. 방법은 다음과 같습니다.

1. 왼쪽의 파일 탐색기에서 작업 중인 노트북을 마우스 오른쪽 단추로 클릭합니다. 

2. **다운로드**를 선택하고 파일을 로컬에 저장합니다.  

이 작업을 수행하면 현재 노트북이 컴퓨터의 기본 다운로드 폴더로 다운로드됩니다.

# 반복 II

# 4단계: 특성 엔지니어링

이제 모델을 훈련하고 평가하는 과정을 한 번 반복했습니다. 모델을 사용해 도달한 첫 번째 결과로는 비즈니스 문제를 해결하는 데 충분하지 않을 수 있다는 점을 감안할 때 모델 성능을 개선하기 위해 데이터에 대해 무엇을 변경할 수 있나요?

### 고려해야 할 주요 질문

1. 두 개의 주요 클래스(*지연* 및 *지연 아님*)의 균형이 모델 성능에 어떤 영향을 미치나요?
2. 상관 관계가 있는 특성이 있나요?
3. 이 단계에서 모델 성능에 긍정적인 영향을 미칠 수 있는 특성 축소 기법이 있나요? 
4. 데이터 또는 데이터 세트를 더 추가하는 것이 좋을까요?
5. 약간의 특성 엔지니어링을 수행한 후 첫 번째 반복과 비교하여 모델 성능이 어떤가요?

다음 셀을 사용하여 모델 성능을 향상할 수 있다고 생각되는 특정 특성 엔지니어링 기법(위의 질문을 가이드로 사용하여)을 수행합니다. 필요에 따라 셀을 삽입하고 삭제합니다.

#### <span style="color: blue;">프로젝트 프레젠테이션: 프로젝트 프레젠테이션에서 이 섹션에서 사용하는 주요 결정과 메서드를 기록합니다. 또한 모델을 다시 평가한 후 얻을 수 있는 새로운 성능 지표도 포함합니다.</span>

시작하기 전에 왜 정밀도 및 재현율은 약 80%인데 정확도는 99%인지 생각해 보십시오.

더 많은 특성 추가

1. 공휴일
2. 날씨

2014년부터 2018까지의 공휴일 목록이 알려져 있기 때문에 표시 변수 **is_holiday**를 생성하여 이를 표시할 수 있습니다.

다른 날보다 공휴일에 비행기 지연율이 더 높을 수 있다는 가설입니다. 2014~2018년의 공휴일이 포함된 부울 변수 `is_holiday`를 추가합니다.

In [ ]:
# Source: http://www.calendarpedia.com/holidays/federal-holidays-2014.html

holidays_14 = ['2014-01-01',  '2014-01-20', '2014-02-17', '2014-05-26', '2014-07-04', '2014-09-01', '2014-10-13', '2014-11-11', '2014-11-27', '2014-12-25' ] 
holidays_15 = ['2015-01-01',  '2015-01-19', '2015-02-16', '2015-05-25', '2015-06-03', '2015-07-04', '2015-09-07', '2015-10-12', '2015-11-11', '2015-11-26', '2015-12-25'] 
holidays_16 = ['2016-01-01',  '2016-01-18', '2016-02-15', '2016-05-30', '2016-07-04', '2016-09-05', '2016-10-10', '2016-11-11', '2016-11-24', '2016-12-25', '2016-12-26']
holidays_17 = ['2017-01-02', '2017-01-16', '2017-02-20', '2017-05-29' , '2017-07-04', '2017-09-04' ,'2017-10-09', '2017-11-10', '2017-11-23', '2017-12-25']
holidays_18 = ['2018-01-01', '2018-01-15', '2018-02-19', '2018-05-28' , '2018-07-04', '2018-09-03' ,'2018-10-08', '2018-11-12','2018-11-22', '2018-12-25']
holidays = holidays_14+ holidays_15+ holidays_16 + holidays_17+ holidays_18

### Add indicator variable for holidays
data_orig['is_holiday'] = # Enter your code here 

기상 데이터는 https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&amp;stations=USW00023174,USW00012960,USW00003017,USW00094846,USW00013874,USW00023234,USW00003927,USW00023183,USW00013881&amp;dataTypes=AWND,PRCP,SNOW,SNWD,TAVG,TMIN,TMAX&amp;startDate=2014-01-01&amp;endDate=2018-12-31에서 가져왔습니다.
<br>

이 데이터 세트에는 공항 코드별로 도시의 풍속, 강수, 강설 및 온도에 대한 정보가 있습니다.

**질문**: 비, 폭풍 또는 눈으로 인한 악천후로 인하여 비행기가 지연될 수 있나요? 이제 확인해보겠습니다.

In [ ]:
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMLFO-1/flight_delay_project/data2/daily-summaries.csv /home/ec2-user/SageMaker/project/data/
#!wget 'https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=USW00023174,USW00012960,USW00003017,USW00094846,USW00013874,USW00023234,USW00003927,USW00023183,USW00013881&dataTypes=AWND,PRCP,SNOW,SNWD,TAVG,TMIN,TMAX&startDate=2014-01-01&endDate=2018-12-31' -O /home/ec2-user/SageMaker/project/data/daily-summaries.csv

데이터 세트의 공항 코드별로 준비된 날씨 데이터를 가져옵니다. 분석에 다음 스테이션 및 공항을 사용합니다. 기상 관측대를 공항 이름에 매핑하는 *공항*이라는 새 열을 만듭니다.

In [ ]:
weather = pd.read_csv('/home/ec2-user/SageMaker/project/data/daily-summaries.csv')
station = ['USW00023174','USW00012960','USW00003017','USW00094846','USW00013874','USW00023234','USW00003927','USW00023183','USW00013881'] 
airports = ['LAX', 'IAH', 'DEN', 'ORD', 'ATL', 'SFO', 'DFW', 'PHX', 'CLT']

### Map weather stations to airport code
station_map = {s:a for s,a in zip(station, airports)}
weather['airport'] = weather['STATION'].map(station_map)

**DATE** 열에서 *MONTH*라는 다른 열을 생성합니다.

In [ ]:
weather['MONTH'] = weather['DATE'].apply(lambda x: x.split('-')[1])
weather.head()

### 샘플 출력
```
  STATION     DATE      AWND PRCP SNOW SNWD TAVG TMAX  TMIN airport MONTH
0 USW00023174 2014-01-01 16   0   NaN  NaN 131.0 178.0 78.0  LAX    01
1 USW00023174 2014-01-02 22   0   NaN  NaN 159.0 256.0 100.0 LAX    01
2 USW00023174 2014-01-03 17   0   NaN  NaN 140.0 178.0 83.0  LAX    01
3 USW00023174 2014-01-04 18   0   NaN  NaN 136.0 183.0 100.0 LAX    01
4 USW00023174 2014-01-05 18   0   NaN  NaN 151.0 244.0 83.0  LAX    01
```

`fillna()`를 사용하여 **SNOW** 및 **SNWD** 열의 결측치를 분석하고 처리합니다. `isna()` 함수를 사용하여 모든 열의 결측치를 확인합니다.

In [ ]:
weather.SNOW.fillna(0, inplace=True)
weather.SNWD.fillna(0, inplace=True)
weather.isna().sum()

**질문**: *TAVG*, *TMAX*, *TMIN*에 대한 결측치가 있는 행의 인덱스를 출력합니다.

**힌트**: 누락된 행을 찾으려면 `isna()` 함수를 사용합니다. 그런 다음 인덱스를 가져 오려면 *idx* 변수의 리스트를 사용하십시오.

In [ ]:
idx = np.array([i for i in range(len(weather))])
TAVG_idx = idx[weather.TAVG.isna()] 
TMAX_idx = # Enter your code here 
TMIN_idx = # Enter your code here 
TAVG_idx

### 샘플 출력

```
array([ 3956,  3957,  3958,  3959,  3960,  3961,  3962,  3963,  3964,
        3965,  3966,  3967,  3968,  3969,  3970,  3971,  3972,  3973,
        3974,  3975,  3976,  3977,  3978,  3979,  3980,  3981,  3982,
        3983,  3984,  3985,  4017,  4018,  4019,  4020,  4021,  4022,
        4023,  4024,  4025,  4026,  4027,  4028,  4029,  4030,  4031,
        4032,  4033,  4034,  4035,  4036,  4037,  4038,  4039,  4040,
        4041,  4042,  4043,  4044,  4045,  4046,  4047, 13420])
```

누락된 *TAVG*, *TMAX* 및 *TMIN* 값을 특정 관측소 또는 공항의 평균값으로 대체할 수 있습니다. *TAVG_idx*의 연속 행이 누락되었기 때문에 이를 이전 값으로 대체하는 것은 불가능합니다. 대신 평균값으로 대체합니다. `groupby` 함수를 사용하여 평균값으로 변수를 집계합니다.

**힌트: ** `MONTH` 및 `STATION`으로 그룹화합니다.

In [ ]:
weather_impute = weather.groupby([<CODE>]).agg({'TAVG':'mean','TMAX':'mean', 'TMIN':'mean' }).reset_index()# Enter your code here
weather_impute.head(2)

평균 데이터를 기상 데이터와 병합합니다.

In [ ]:

weather = pd.merge(weather, weather_impute,  how='left', left_on=['MONTH','STATION'], right_on = ['MONTH','STATION'])\
.rename(columns = {'TAVG_y':'TAVG_AVG',
                   'TMAX_y':'TMAX_AVG', 
                   'TMIN_y':'TMIN_AVG',
                   'TAVG_x':'TAVG',
                   'TMAX_x':'TMAX', 
                   'TMIN_x':'TMIN'})

결측치가 있는지 다시 확인합니다.

In [ ]:
weather.TAVG[TAVG_idx] = weather.TAVG_AVG[TAVG_idx]
weather.TMAX[TMAX_idx] = weather.TMAX_AVG[TMAX_idx]
weather.TMIN[TMIN_idx] = weather.TMIN_AVG[TMIN_idx]
weather.isna().sum()

`STATION,MONTH,TAVG_AVG,TMAX_AVG,TMIN_AVG,TMAX,TMIN,SNWD`를 데이터 세트에서 삭제합니다.

In [ ]:
weather.drop(columns=['STATION','MONTH','TAVG_AVG', 'TMAX_AVG', 'TMIN_AVG', 'TMAX' ,'TMIN', 'SNWD'],inplace=True)

출발지 및 목적지 기상 상태를 데이터 세트에 추가합니다.

In [ ]:
### Add origin weather conditions
data_orig = pd.merge(data_orig, weather,  how='left', left_on=['FlightDate','Origin'], right_on = ['DATE','airport'])\
.rename(columns = {'AWND':'AWND_O','PRCP':'PRCP_O', 'TAVG':'TAVG_O', 'SNOW': 'SNOW_O'})\
.drop(columns=['DATE','airport'])

### Add destination weather conditions
data_orig = pd.merge(data_orig, weather,  how='left', left_on=['FlightDate','Dest'], right_on = ['DATE','airport'])\
.rename(columns = {'AWND':'AWND_D','PRCP':'PRCP_D', 'TAVG':'TAVG_D', 'SNOW': 'SNOW_D'})\
.drop(columns=['DATE','airport'])

**참고**: 조인 후에는 null 또는 NA를 확인하는 것이 좋습니다.

In [ ]:
sum(data.isna().any())

In [ ]:
data_orig.columns

원-핫 인코딩을 사용하여 범주형 데이터를 숫자형 데이터로 변환합니다.

In [ ]:
data = data_orig.copy()
data = data[['is_delay', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest','Distance','DepHourofDay','is_holiday', 'AWND_O', 'PRCP_O',
       'TAVG_O', 'AWND_D', 'PRCP_D', 'TAVG_D', 'SNOW_O', 'SNOW_D']]


categorical_columns  = ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest', 'is_holiday']
for c in categorical_columns:
    data[c] = data[c].astype('category')

In [ ]:
data_dummies = pd.get_dummies(data[['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest', 'is_holiday']], drop_first=True)
data = pd.concat([data, data_dummies], axis = 1)
data.drop(categorical_columns,axis=1, inplace=True)

새로운 열을 확인합니다.

In [ ]:
data.shape

In [ ]:
data.columns

### 샘플 출력

```
Index(['Distance', 'DepHourofDay', 'is_delay', 'AWND_O', 'PRCP_O', 'TAVG_O',
       'AWND_D', 'PRCP_D', 'TAVG_D', 'SNOW_O', 'SNOW_D', 'Year_2015',
       'Year_2016', 'Year_2017', 'Year_2018', 'Quarter_2', 'Quarter_3',
       'Quarter_4', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12',
       'DayofMonth_2', 'DayofMonth_3', 'DayofMonth_4', 'DayofMonth_5',
       'DayofMonth_6', 'DayofMonth_7', 'DayofMonth_8', 'DayofMonth_9',
       'DayofMonth_10', 'DayofMonth_11', 'DayofMonth_12', 'DayofMonth_13',
       'DayofMonth_14', 'DayofMonth_15', 'DayofMonth_16', 'DayofMonth_17',
       'DayofMonth_18', 'DayofMonth_19', 'DayofMonth_20', 'DayofMonth_21',
       'DayofMonth_22', 'DayofMonth_23', 'DayofMonth_24', 'DayofMonth_25',
       'DayofMonth_26', 'DayofMonth_27', 'DayofMonth_28', 'DayofMonth_29',
       'DayofMonth_30', 'DayofMonth_31', 'DayOfWeek_2', 'DayOfWeek_3',
       'DayOfWeek_4', 'DayOfWeek_5', 'DayOfWeek_6', 'DayOfWeek_7',
       'Reporting_Airline_DL', 'Reporting_Airline_OO', 'Reporting_Airline_UA',
       'Reporting_Airline_WN', 'Origin_CLT', 'Origin_DEN', 'Origin_DFW',
       'Origin_IAH', 'Origin_LAX', 'Origin_ORD', 'Origin_PHX', 'Origin_SFO',
       'Dest_CLT', 'Dest_DEN', 'Dest_DFW', 'Dest_IAH', 'Dest_LAX', 'Dest_ORD',
       'Dest_PHX', 'Dest_SFO', 'is_holiday_1'],
      dtype='object')
```

**is_delay** 열의 이름을 다시 *target*으로 바꿉니다. 이전에 사용한 것과 동일한 코드를 사용하십시오.

In [ ]:
data.rename(columns = {<CODE>:<CODE>}, inplace=True )# Enter your code here

훈련 세트를 다시 생성합니다.

**힌트: ** 이전에 정의 (및 사용한) `split_data` 함수를 사용하십시오.

In [ ]:
# Enter your code here

### 새로운 기준 분류자

이제 이러한 새로운 특성으로 모델의 예측 성능이 향상되는지 확인합니다.

In [ ]:
# Instantiate the LinearLearner estimator object
classifier_estimator2 = # Enter your code here

### 샘플 코드

```
num_classes = len(pd.unique(train_labels)) 
classifier_estimator2 = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                               instance_count=1,
                                               instance_type='ml.m4.xlarge',
                                               predictor_type='binary_classifier',
                                               binary_classifier_model_selection_criteria = 'cross_entropy_loss')
```

In [ ]:
train_records = classifier_estimator2.record_set(train.values[:, 1:].astype(np.float32), train.values[:, 0].astype(np.float32), channel='train')
val_records = classifier_estimator2.record_set(validate.values[:, 1:].astype(np.float32), validate.values[:, 0].astype(np.float32), channel='validation')
test_records = classifier_estimator2.record_set(test.values[:, 1:].astype(np.float32), test.values[:, 0].astype(np.float32), channel='test')

방금 만든 세 개의 데이터 세트를 사용하여 모델을 훈련합니다.

In [ ]:
# Enter your code here

새로 학습된 모델을 사용하여 배치 예측을 수행합니다.

In [ ]:
# Enter your code here

혼동 행렬을 플롯합니다.

In [ ]:
# Enter your code here

ROC 곡선을 플롯합니다.

In [ ]:
# Enter your code here

선형 모델은 성능이 약간 향상되었을 뿐입니다. Amazon SageMaker에서 *XGBoost*라는 트리 기반 앙상블 모델을 사용해 보십시오.

### XGBoost 모델 사용해 보기

다음 단계를 수행합니다.  

1. 훈련 세트 변수를 사용하여 train.csv, validation.csv 및 test.csv 등의 CSV 파일로 저장합니다.
2. 변수에 버킷 이름을 저장합니다. Amazon S3 버킷 이름은 실습 지침의 왼쪽에 나와 있습니다.  
a. `bucket = <LabBucketName>`  
b. `prefix = 'flight-xgb'`  
3. Python용 AWS SDK(Boto3)를 사용하여 모델을 버킷에 업로드합니다.    

In [ ]:
bucket='c218151a5506212l17131779t1w715054373660-labbucket-ntokzfktykpe'
prefix='flight-xgb'
train_file='flight_train.csv'
test_file='flight_test.csv'
validate_file='flight_validate.csv'
whole_file='flight.csv'
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False )
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

`sagemaker.inputs.TrainingInput` 함수를 사용하여 훈련 및 검증 데이터 세트에 `record_set`를 생성합니다.

In [ ]:
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
from sagemaker.image_uris import retrieve
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

In [ ]:
sess = sagemaker.Session()
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

xgb = sagemaker.estimator.Estimator(container,
                                    role = sagemaker.get_execution_role(), 
                                    instance_count=1, 
                                    instance_type=instance_type,
                                    output_path=s3_output_location,
                                    sagemaker_session=sess)
xgb.set_hyperparameters(max_depth=5,
                        eta=0.2,
                        gamma=4,
                        min_child_weight=6,
                        subsample=0.8,
                        silent=0,
                        objective='binary:logistic',
                        eval_metric = "auc", 
                        num_round=100)

xgb.fit(inputs=data_channels)

새 모델에 배치 변환기를 사용하고 테스트 데이터 세트에서 모델을 평가합니다.

In [ ]:
batch_X = test.iloc[:,1:];
batch_X_file='batch-in.csv'
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = xgb.transformer(instance_count=1,
                                       instance_type=instance_type,
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

예측 대상 및 테스트 레이블을 가져옵니다.

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),',',names=['target'])
test_labels = test.iloc[:,0]

정의된 임계값을 기반으로 예측된 값을 계산합니다.

**참고: ** 예측 대상은 점수로 나타나며, 이진 클래스로 변환해야 합니다.

In [ ]:
print(target_predicted.head())

def binary_convert(x):
    threshold = 0.55
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['target'] = target_predicted['target'].apply(binary_convert)

test_labels = test.iloc[:,0]

print(target_predicted.head())

`target_predicted` 및 `test_labels`에 대한 혼동 행렬을 플롯합니다.

In [ ]:
# Enter your code here

ROC 도표를 플롯합니다.

In [ ]:
# Enter your code here

### 다른 임계값 시도

**질문**: 모델이 테스트 세트를 얼마나 잘 처리했는지에 따라 어떤 결론을 내릴 수 있습니까?

In [ ]:
#Enter your answer here

### 하이퍼파라미터 최적화(HPO)

In [ ]:
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner

### You can spin up multiple instances to do hyperparameter optimization in parallel

xgb = sagemaker.estimator.Estimator(container,
                                    role=sagemaker.get_execution_role(), 
                                    instance_count= 1, # make sure you have a limit set for these instances
                                    instance_type=instance_type, 
                                    output_path='s3://{}/{}/output'.format(bucket, prefix),
                                    sagemaker_session=sess)

xgb.set_hyperparameters(eval_metric='auc',
                        objective='binary:logistic',
                        num_round=100,
                        rate_drop=0.3,
                        tweedie_variance_power=1.4)

hyperparameter_ranges = {'alpha': ContinuousParameter(0, 1000, scaling_type='Linear'),
                         'eta': ContinuousParameter(0.1, 0.5, scaling_type='Linear'),
                         'min_child_weight': ContinuousParameter(3, 10, scaling_type='Linear'),
                         'subsample': ContinuousParameter(0.5, 1),
                         'num_round': IntegerParameter(10,150)}

objective_metric_name = 'validation:auc'

tuner = HyperparameterTuner(xgb,
                            objective_metric_name,
                            hyperparameter_ranges,
                            max_jobs=10, # Set this to 10 or above depending upon budget and available time.
                            max_parallel_jobs=1)

In [ ]:
tuner.fit(inputs=data_channels)
tuner.wait()

<i class="fas fa-exclamation-triangle" style="color:red"></i>훈련 작업이 완료될 때까지 기다립니다. 25~30분 정도 걸릴 수 있습니다.

**하이퍼파라미터 최적화 작업을 모니터링하려면 다음 순서를 따르십시오.**  

1. AWS Management Console의 **Services** 메뉴에서 **Amazon SageMaker**를 선택합니다.  
2. **훈련 > 하이퍼파라미터 튜닝 작업**을 선택합니다.
3. 각 하이퍼파라미터 튜닝 작업의 상태, 목표 지표 값 및 로그를 확인할 수 있습니다.  

작업이 성공적으로 완료되었는지 확인합니다.

In [ ]:
boto3.client('sagemaker').describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name)['HyperParameterTuningJobStatus']

하이퍼파라미터 튜닝 작업을 마치면 가장 잘 작동하는 모델이 있을 것입니다. 튜닝 작업에서 해당 모델에 대한 정보를 얻을 수 있습니다.

In [ ]:
sage_client = boto3.Session().client('sagemaker')
tuning_job_name = tuner.latest_tuning_job.job_name
print(f'tuning job name:{tuning_job_name}')
tuning_job_result = sage_client.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=tuning_job_name)
best_training_job = tuning_job_result['BestTrainingJob']
best_training_job_name = best_training_job['TrainingJobName']
print(f"best training job: {best_training_job_name}")

best_estimator = tuner.best_estimator()

tuner_df = sagemaker.HyperparameterTuningJobAnalytics(tuning_job_name).dataframe()
tuner_df.head()

`best_estimator` 추정기를 사용하고 데이터를 사용하여 훈련합니다. 

**팁:** 이전 XGBoost 추정기 맞춤 기능을 참조하십시오.

In [ ]:
# Enter your code here'

새 모델에 배치 변환기를 사용하고 테스트 데이터 세트에서 모델을 평가합니다.

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = best_estimator.transformer(instance_count=1,
                                       instance_type=instance_type,
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),',',names=['target'])
test_labels = test.iloc[:,0]

예측 대상 및 테스트 레이블을 가져옵니다.

In [ ]:
print(target_predicted.head())

def binary_convert(x):
    threshold = 0.55
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['target'] = target_predicted['target'].apply(binary_convert)

test_labels = test.iloc[:,0]

print(target_predicted.head())

`target_predicted` 및 `test_labels`에 대한 혼동 행렬을 플롯합니다.

In [ ]:
# Enter your code here

ROC 도표를 플롯합니다.

In [ ]:
# Enter your code here

**질문**: 다른 하이퍼파라미터와 하이퍼파라미터 범위를 시도합니다. 이러한 변경으로 인해 모델이 개선됩니까?

## 결론

이제 최소한 두 번 이상 모델을 훈련하고 평가하는 과정을 반복했습니다. 이 프로젝트를 마무리하고 다음 사항을 되짚어 볼 때입니다.

- 지금까지 배운 것 
- (시간 여유가 있다고 가정할 때) 앞으로 더 수행할 수 있는 단계 유형

다음 셀을 사용하여 이러한 질문 및 기타 관련 질문에 답합니다.

1. 모델 성능이 비즈니스 목표에 부합하나요? 그렇지 않은 경우 튜닝할 시간이 더 있었다면 어떻게 다르게 할 수 있을까요?
2. 데이터 세트, 특성 및 하이퍼파라미터를 변경함에 따라 모델이 얼마나 향상되었나요? 이 프로젝트 전체에서 모델을 가장 크게 개선한 것으로 생각되는 기법 유형은 무엇인가요?
3. 이 프로젝트를 통틀어 가장 어려움을 겪었던 문제에는 어떤 것들이 있나요?
4. 파이프라인에서 여러분이 이해할 수 없었던 부분 중 답을 찾지 못한 질문이 있나요?
5. 이 프로젝트를 작업하면서 기계 학습에 대해 배운 가장 중요한 세 가지는 무엇인가요?

#### <span style="color: blue;">프로젝트 프레젠테이션: 이러한 질문에 대한 답변을 요약하여 프로젝트 프레젠테이션에 또한 추가합니다. 프로젝트 프레젠테이션을 위한 모든 노트를 취합하여 학급을 대상으로 발표할 준비를 합니다.</span>

In [ ]:
# Write your answers here